# Graph-Enriched Search

In the previous notebook (`01_intro_strands_mcp.ipynb`), you connected a Strands Agent to the Neo4j MCP Server and inspected the graph schema. Now you'll combine **vector search** with **graph traversal** using pre-written Cypher queries — the **Cypher Templates** pattern.

After finding semantically similar chunks, the agent traverses graph relationships to pull in document metadata, neighboring chunks, and connected entities like companies, products, and risk factors. The graph turns text retrieval into knowledge retrieval.

**Learning Objectives:**
- Understand how graph traversal enriches vector search results with structured context
- Walk through two levels of graph traversal — document context and entity context
- Combine vector similarity search with Cypher graph traversal
- Use the **Cypher Templates** `@tool` pattern to encapsulate embeddings and queries

**How it works:**
1. Custom `@tool` wrappers encapsulate embedding generation and Cypher execution — the LLM never sees raw float arrays
2. Each tool runs a pre-written Cypher query that performs vector search to find matching chunks via MCP
3. For each match, the Cypher query traverses graph relationships to gather additional context:
   - Previous and next chunks (for a wider text window)
   - Source document metadata
   - Connected entities (companies, risk factors, etc.)
4. The enriched results give the LLM more context for generating answers

This mirrors the `VectorCypherRetriever` from the neo4j-graphrag library, but executed through MCP with pre-written Cypher templates.

**Prerequisites:**
- `../CONFIG.txt` configured with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

> The database, embeddings, and indexes are already set up via the pre-deployed MCP server. You do not need to load data yourself.

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx -q

## 1. Configuration and Imports

In [ ]:
import os
import uuid

from dotenv import load_dotenv
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent, tool
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

load_dotenv("../CONFIG.txt")

MODEL_ID = os.getenv("MODEL_ID")
REGION = os.getenv("REGION", "us-east-1")
MCP_GATEWAY_URL = os.getenv("MCP_GATEWAY_URL")
MCP_ACCESS_TOKEN = os.getenv("MCP_ACCESS_TOKEN")

assert MCP_GATEWAY_URL, "MCP_GATEWAY_URL not configured in CONFIG.txt"
assert MCP_ACCESS_TOKEN, "MCP_ACCESS_TOKEN not configured in CONFIG.txt"

print(f"Model:     {MODEL_ID}")
print(f"Region:    {REGION}")
print("Configuration loaded!")

## 2. Shared Utilities and MCP Connection

The embedding helper from `lib/` centralizes Bedrock embedding configuration. The MCP connection is opened persistently for the notebook session, and the Cypher query tool is discovered automatically.

### Entity Relationships in the Graph

The knowledge graph connects chunks to companies through document ownership:

```
(Chunk)-[:FROM_DOCUMENT]->(Document)<-[:FILED]-(Company)  the company that filed the document
(Product)-[:FROM_CHUNK]->(Chunk)     products mentioned in a chunk
(Company)-[:FACES_RISK]->(RiskFactor)  risks associated with companies
```

By traversing these relationships, the agent can augment each vector match with structured entity data.

In [ ]:
from botocore.config import Config as BotocoreConfig
from lib.lab_5_data_utils import get_embedding

# Verify the shared embedding function works
test_embedding = get_embedding("test")
print(f"Embedding dimensions: {len(test_embedding)}")

# Model
bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
    boto_client_config=BotocoreConfig(read_timeout=300),
)

# Open a persistent MCP connection for the notebook session
mcp_client = MCPClient(lambda: streamablehttp_client(
    url=MCP_GATEWAY_URL,
    headers={"Authorization": f"Bearer {MCP_ACCESS_TOKEN}"},
))
mcp_client.__enter__()

# Discover available tools and find the Cypher query tool
mcp_tools = mcp_client.list_tools_sync()
tool_names = [t.tool_name for t in mcp_tools]
print(f"MCP tools discovered: {tool_names}")

cypher_tool = next((n for n in tool_names if "read-cypher" in n), None)
assert cypher_tool, f"No Cypher query tool found among: {tool_names}"

print(f"\nModel: {MODEL_ID}")
print(f"Cypher tool: {cypher_tool}")

## 3. Graph-Enriched Search (Cypher Template)

A basic vector search returns isolated chunks — text fragments ranked by similarity with no surrounding context. Graph-enriched search fixes that by traversing relationships after the vector match.

The `@tool` function below is a **Cypher Template** — a pre-written, expert-reviewed query that the agent calls without modification. For each matched chunk, the Cypher follows two relationship types:

```
(prev:Chunk)-[:NEXT_CHUNK]->(node)-[:NEXT_CHUNK]->(next:Chunk)   neighboring chunks
(node)-[:FROM_DOCUMENT]->(doc:Document)                           source document
```

Because filings are split into chunks, relevant information often spans a chunk boundary. The `NEXT_CHUNK` traversal recovers that lost context by pulling in the previous and next chunks, giving the LLM a wider text window. The `FROM_DOCUMENT` traversal adds the source document name for citation — context that pure vector search cannot provide.

In [ ]:
@tool
def graph_enriched_search(query: str, top_k: int = 3) -> str:
    """Search for similar chunks enriched with document and neighboring chunk context.
    Returns chunk text, source document, and text from adjacent chunks."""
    embedding = get_embedding(query)
    top_k = int(top_k)

    cypher = f"""
        CALL db.index.vector.queryNodes('chunkEmbeddings', {top_k}, $query_vector)
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
        OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
        WITH node, doc, score,
             CASE WHEN prev IS NOT NULL THEN prev.text ELSE '' END AS prev_text,
             CASE WHEN next IS NOT NULL THEN next.text ELSE '' END AS next_text
        WITH node {{.*, embedding: null}} AS node, doc, score, prev_text, next_text
        RETURN node.text AS text, score, doc.name AS document,
               prev_text AS previous_chunk, next_text AS next_chunk
        ORDER BY score DESC
    """
    result = mcp_client.call_tool_sync(
        tool_use_id=str(uuid.uuid4()),
        name=cypher_tool,
        arguments={"query": cypher, "params": {"query_vector": embedding}},
    )
    return result["content"][0]["text"]


GRAPH_ENRICHED_PROMPT = """You are a retrieval assistant that performs graph-enriched vector search against a Neo4j database containing SEC 10-K filing data.

You have a graph_enriched_search tool that finds similar chunks and enriches them with the source document name and neighboring chunk text for additional context.

For each result, show:
- Similarity score
- Source document name
- The matched chunk text
- Context from neighboring chunks (if available)"""

graph_agent = Agent(
    model=bedrock_model,
    system_prompt=GRAPH_ENRICHED_PROMPT,
    tools=[graph_enriched_search],
)

print("Graph-enriched search ready!")
print(graph_agent(f"Search for: What financial metrics indicate company performance?\nUse top_k=3."))

## 4. Entity-Enriched Search (Cypher Template)

Graph-enriched search added document names and neighboring chunks. Entity-enriched search goes further — it traverses from chunks all the way to the structured entities in the knowledge graph:

```
(node)-[:FROM_DOCUMENT]->(doc:Document)<-[:FILED]-(company:Company)   who filed it
(company)-[:FACES_RISK]->(risk:RiskFactor)                            risks they face
(product:Product)-[:FROM_CHUNK]->(node)                               products mentioned
```

This is a second **Cypher Template** — another pre-written query, but one that crosses from the unstructured layer (chunks) into the structured layer (entities). Now the LLM receives not just text and context, but the graph's understanding of *what companies, products, and risks* are connected to each chunk. This is what makes retrieval from a knowledge graph different from a standalone vector database — the structured entity layer sitting on top of the text.

In [ ]:
@tool
def entity_enriched_search(query: str, top_k: int = 3) -> str:
    """Search for similar chunks enriched with companies, products, and risk factors.
    Returns chunk text, source document, and connected entities."""
    embedding = get_embedding(query)
    top_k = int(top_k)

    cypher = f"""
        CALL db.index.vector.queryNodes('chunkEmbeddings', {top_k}, $query_vector)
        YIELD node, score
        MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
        OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
        OPTIONAL MATCH (company)-[:FACES_RISK]->(risk:RiskFactor)
        WITH node, doc, score,
             collect(DISTINCT company.name) AS companies,
             collect(DISTINCT risk.name)[0..5] AS risks
        OPTIONAL MATCH (product:Product)-[:FROM_CHUNK]->(node)
        WITH node, doc, score, companies, risks,
             collect(DISTINCT product.name)[0..5] AS products
        WITH node {{.*, embedding: null}} AS node, doc, score, companies, risks, products
        RETURN node.text AS text, score, doc.name AS document,
               companies, risks, products
        ORDER BY score DESC
    """
    result = mcp_client.call_tool_sync(
        tool_use_id=str(uuid.uuid4()),
        name=cypher_tool,
        arguments={"query": cypher, "params": {"query_vector": embedding}},
    )
    return result["content"][0]["text"]


ENTITY_ENRICHED_PROMPT = """You are a retrieval assistant that performs entity-enriched vector search against a Neo4j database containing SEC 10-K filing data.

You have an entity_enriched_search tool that finds similar chunks and enriches them with connected companies, products, and risk factors from the knowledge graph.

For each result, show:
- Similarity score
- Source document name
- The matched chunk text
- Companies, products, and risk factors connected to the chunk"""

entity_agent = Agent(
    model=bedrock_model,
    system_prompt=ENTITY_ENRICHED_PROMPT,
    tools=[entity_enriched_search],
)

print("Entity-enriched search ready!")
print(entity_agent(f"Search for: What financial metrics indicate company performance?\nUse top_k=3."))

## 5. Entity-Enriched Q&A

Sections 3 and 4 displayed raw retrieval results — chunks, scores, and connected data. Now use the entity-enriched agent as a full question-answering pipeline. The agent finds relevant chunks via vector search, gathers graph context using the Cypher Template, and synthesizes a comprehensive answer from all of it.

In [ ]:
QA_PROMPT = """You are a financial analysis assistant with access to a Neo4j knowledge graph containing SEC 10-K filing data.

You have an entity_enriched_search tool that searches for relevant chunks enriched with companies, products, and risk factors.

After retrieving results, synthesize a clear answer based on the retrieved context.
Include the companies, products, and risk factors found. Cite the source documents."""

qa_agent = Agent(
    model=bedrock_model,
    system_prompt=QA_PROMPT,
    tools=[entity_enriched_search],
)


def ask(query: str, top_k: int = 5):
    """Ask a question using entity-enriched vector search for context."""
    print(f'Question: "{query}"')
    print("-" * 60)

    response = qa_agent(
        f"Answer this question using entity-enriched search with top_k={top_k}.\n\n"
        f"Question: {query}"
    )
    print(f"\n{response}")
    return response


print("Q&A function ready!")

In [ ]:
result = ask("What are the key risk factors mentioned in Apple's 10-K filing?")

In [ ]:
result = ask("Which companies face cybersecurity-related risks?")

## Summary

You walked through two retrieval approaches that show the power of graph-enriched search:

| Approach | What the LLM Receives |
|----------|----------------------|
| **Graph-enriched** | The matched chunk + document name + neighboring chunk text |
| **Entity-enriched** | The matched chunk + document + companies, products, risk factors |

Each level of graph traversal adds more context. The entity-enriched approach is the most powerful because it connects unstructured text (chunks) to structured data (entity nodes and their relationships), giving the LLM both the raw filing text and the knowledge graph's understanding of what's in it.

### Cypher Templates vs. Text2Cypher

This notebook used the **Cypher Templates** pattern — every query was pre-written in a `@tool` function. This is reliable and predictable, but limited to the query patterns you define in advance.

The next notebook takes the opposite approach: the agent discovers the schema and writes original Cypher from scratch — the **Text2Cypher** pattern. More flexible, but query quality depends on the LLM's ability to reason about graph structures.

### Embedding Handling

Embeddings are kept off the LLM context window at two levels:

1. **Input path**: `@tool` wrappers encapsulate embedding generation — the agent calls `graph_enriched_search("query")`, never seeing the 1024-dim float array
2. **Output path**: Cypher uses `WITH node {.*, embedding: null}` to strip embeddings before returning results, plus explicit property selection in the RETURN clause

---

**Next:** [Text2Cypher Agent](03_text2cypher_agent.ipynb) — the agent writes its own Cypher from scratch